# 05 - Feature Engineering

**Objetivo**: Construir la matriz de ~35 features a partir de los datos recopilados.

Features por grupo:
1. **Tokenomics** (6): Concentracion de holders, supply
2. **Liquidez** (6): Profundidad, crecimiento, estabilidad
3. **Price Action** (12): Retornos, volatilidad, tendencias
4. **Social** (7): Buyers/sellers, tamano de TX
5. **Contrato** (3): Verificacion, ownership, edad
6. **Contexto** (7): Estado del mercado, timing

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.data.storage import Storage
from src.features.builder import FeatureBuilder
import config

storage = Storage()
print(f"Tokens en BD: {storage.stats().get('tokens', 0)}")

## Construir features con FeatureBuilder

In [ ]:
# Crear el builder
builder = FeatureBuilder(storage=storage)

# Construir features para TODOS los tokens en la BD
# Esto puede tomar unos minutos dependiendo de la cantidad de tokens
features_df = builder.build_all_features()

print(f"\nMatriz de features construida:")
print(f"  Tokens: {len(features_df)}")
print(f"  Features: {len(features_df.columns)}")
print(f"\nColumnas: {features_df.columns.tolist()}")

In [ ]:
# Ver las primeras filas
if not features_df.empty:
    display(features_df.head())

In [ ]:
# Estadisticas descriptivas de cada feature
if not features_df.empty:
    # Solo columnas numericas
    numeric_features = features_df.select_dtypes(include=[np.number])
    stats = numeric_features.describe().T
    stats['null_pct'] = numeric_features.isnull().mean() * 100
    display(stats.round(3))

## Analisis de calidad de features

In [ ]:
if not features_df.empty:
    numeric_features = features_df.select_dtypes(include=[np.number])
    
    # Features con muchos valores nulos (>50%)
    null_pct = numeric_features.isnull().mean() * 100
    high_null = null_pct[null_pct > 50].sort_values(ascending=False)
    
    if not high_null.empty:
        print("⚠️ Features con >50% valores nulos:")
        for feat, pct in high_null.items():
            print(f"  {feat}: {pct:.1f}% nulos")
    else:
        print("Todos los features tienen <50% nulos")
    
    # Features con varianza 0 (constantes)
    zero_var = numeric_features.std() == 0
    const_features = zero_var[zero_var].index.tolist()
    if const_features:
        print(f"\n⚠️ Features constantes (eliminar): {const_features}")
    else:
        print("No hay features constantes")

In [ ]:
# Correlaciones altas entre features (posible redundancia)
if not features_df.empty:
    numeric_features = features_df.select_dtypes(include=[np.number])
    
    if len(numeric_features.columns) >= 2:
        corr = numeric_features.corr()
        
        # Encontrar pares con correlacion > 0.9
        high_corr_pairs = []
        for i in range(len(corr.columns)):
            for j in range(i+1, len(corr.columns)):
                if abs(corr.iloc[i, j]) > 0.9:
                    high_corr_pairs.append((
                        corr.columns[i], 
                        corr.columns[j], 
                        corr.iloc[i, j]
                    ))
        
        if high_corr_pairs:
            print("⚠️ Pares con correlacion > 0.9 (considerar eliminar uno):")
            for f1, f2, c in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
                print(f"  {f1} <-> {f2}: {c:.3f}")
        else:
            print("No hay pares de features altamente correlacionados")

## Guardar features

In [ ]:
if not features_df.empty:
    # Guardar en Parquet
    features_df.to_parquet(
        config.PROCESSED_DIR / "features.parquet", 
        index=True
    )
    print(f"Features guardados: {config.PROCESSED_DIR / 'features.parquet'}")
    
    # Guardar en SQLite tambien
    storage.save_features_df(features_df)
    print("Features guardados en SQLite")
    
    print(f"\nResumen: {len(features_df)} tokens x {len(features_df.columns)} features")
    print("\nSiguiente paso: notebook 06_labeling.ipynb")
else:
    print("No se generaron features. Verifica que hay datos en la BD.")